# 01 — Train Qwen 3.5 4B Text-Only LoRA / QLoRA (Botobot / Group 6)

**Project:** Mapúa University AI3 Machine Project — Philippine Voting Process Helper NLP (`Botobot / Group 6`)  
**Base Model:** `Qwen/Qwen3.5-4B`  
**SFT Dataset:** `data/sft/qwen35/{train,val}.jsonl` (`431` train / `49` val, disjoint `section_key`s, completion-only loss)  
**Precision Modes:** `QUANT_MODE = "8bit"` (LLM.int8 LoRA), `"4bit"` (NF4 QLoRA), or `"16bit"` (BF16 LoRA on ≥ 20/32 GB VRAM)  
**Target Runtime:** Google Colab (`L4 24 GB` primary / `T4 16 GB` fallback)

In [ ]:
# @title Cell 1: Install Dependencies
!pip -q install -U "transformers>=5.5.0" accelerate peft trl bitsandbytes datasets

In [ ]:
# @title Cell 2: Mount Google Drive, Config Switches & GPU Auto-Guard
import json
import os
from pathlib import Path
import torch

# --- Experiment Switches (override via env vars for headless colab exec) ---
MODEL_ID = "Qwen/Qwen3.5-4B"
SFT_SUBDIR = "qwen35"
QUANT_MODE = os.environ.get("AI3_QUANT_MODE", "8bit").lower()  # "8bit", "4bit", or "16bit"
NUM_EPOCHS = int(os.environ.get("AI3_EPOCHS", "3"))
MAX_STEPS = int(os.environ.get("AI3_MAX_STEPS", "-1"))          # -1 = full epochs; >0 = smoke/test run
MAX_LENGTH = int(os.environ.get("AI3_MAX_LENGTH", "1536"))
LEARNING_RATE = float(os.environ.get("AI3_LR", "2e-4"))
SEED = int(os.environ.get("AI3_SEED", "42"))
ADAPTER_TAG = os.environ.get("AI3_ADAPTER_TAG", "")             # optional suffix e.g. "_8bit_test"
USE_DRIVE = os.environ.get("AI3_USE_DRIVE", "1") == "1"

# --- Storage Root (Google Drive in interactive Colab, /content/AI3_Botobot fallback for CLI) ---
if os.environ.get("AI3_BASE_DIR"):
    BASE_DIR = Path(os.environ["AI3_BASE_DIR"])
elif USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/AI3_Botobot")
    except Exception as exc:
        print(f"[WARN] Drive mount unavailable ({exc}); falling back to /content/AI3_Botobot")
        BASE_DIR = Path("/content/AI3_Botobot")
else:
    BASE_DIR = Path("/content/AI3_Botobot")

SFT_DIR = BASE_DIR / "data" / "sft" / SFT_SUBDIR
ADAPTER_DIR = BASE_DIR / "adapters" / f"qwen35_4b_lora{ADAPTER_TAG}"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

# --- GPU Auto-Guard ---
assert torch.cuda.is_available(), "CUDA GPU is required for training. Select a T4/L4/A100 runtime."
GPU_NAME = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if MAJOR >= 8:
    COMPUTE_DTYPE = torch.bfloat16
else:
    COMPUTE_DTYPE = torch.float16
ATTN_IMPL = None if "gemma-4" in MODEL_ID.lower() else "sdpa"

if QUANT_MODE == "16bit" and VRAM_GB < 20:
    print(f"[GUARD] 16-bit LoRA needs >= 20 GB VRAM (detected {VRAM_GB:.1f} GB on {GPU_NAME}). Switching to 8bit.")
    QUANT_MODE = "8bit"

PER_DEVICE_BS = int(os.environ.get("AI3_BATCH_SIZE", "2"))
GRAD_ACCUM = int(os.environ.get("AI3_GRAD_ACCUM", "8"))

HW_META = {
    "model_id": MODEL_ID,
    "gpu_name": GPU_NAME,
    "compute_capability": f"{MAJOR}.{MINOR}",
    "vram_gb": round(VRAM_GB, 2),
    "compute_dtype": str(COMPUTE_DTYPE),
    "attn_implementation": ATTN_IMPL,
    "quant_mode": QUANT_MODE,
    "num_epochs": NUM_EPOCHS,
    "max_steps": MAX_STEPS,
    "max_length": MAX_LENGTH,
    "per_device_batch_size": PER_DEVICE_BS,
    "grad_accum": GRAD_ACCUM,
    "effective_batch_size": PER_DEVICE_BS * GRAD_ACCUM,
    "base_dir": str(BASE_DIR),
    "adapter_dir": str(ADAPTER_DIR),
}
print(json.dumps(HW_META, indent=2))

In [ ]:
# @title Cell 3: Load Pre-Formatted SFT Dataset (Completion-Only Prompt + Completion)
from datasets import load_dataset

train_path = SFT_DIR / "train.jsonl"
val_path = SFT_DIR / "val.jsonl"
assert train_path.exists(), f"Missing {train_path}. Upload data/sft/{SFT_SUBDIR}/train.jsonl first."
assert val_path.exists(), f"Missing {val_path}. Upload data/sft/{SFT_SUBDIR}/val.jsonl first."

raw_ds = load_dataset("json", data_files={"train": str(train_path), "validation": str(val_path)})
keep_cols = ["prompt", "completion"]
train_ds = raw_ds["train"].remove_columns([c for c in raw_ds["train"].column_names if c not in keep_cols])
val_ds = raw_ds["validation"].remove_columns([c for c in raw_ds["validation"].column_names if c not in keep_cols])

assert all(r["prompt"].strip() and r["completion"].strip() for r in train_ds), "Empty prompt/completion in train!"
assert all(r["prompt"].strip() and r["completion"].strip() for r in val_ds), "Empty prompt/completion in val!"

print(f"Loaded train={len(train_ds)} rows, val={len(val_ds)} rows")
print("Sample prompt tail:", repr(train_ds[0]["prompt"][-120:]))
print("Sample completion head:", repr(train_ds[0]["completion"][:160]))

In [ ]:
# @title Cell 4: Load Base Model (Qwen/Qwen3.5-4B) & Freeze Multimodal Towers
import inspect
import logging
import transformers
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# Silence repetitive benign fp32->fp16 cast warnings during 8-bit training/inference
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

print("transformers version:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if QUANT_MODE == "8bit":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
elif QUANT_MODE == "4bit":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
else:
    quant_config = None

load_kwargs = dict(
    quantization_config=quant_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
)
if ATTN_IMPL is not None:
    load_kwargs["attn_implementation"] = ATTN_IMPL

try:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
except Exception as exc:
    print(f"[INFO] AutoModelForImageTextToText fallback ({exc}); using AutoModelForCausalLM")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)

if QUANT_MODE in ("8bit", "4bit"):
    if "gemma-4" in MODEL_ID.lower():
        # Avoid peft's bulk fp16->fp32 upcast on Gemma 4's 5.25 GB Per-Layer Embeddings (+10.5 GB VRAM spike)
        for p in model.parameters():
            p.requires_grad = False
        model.enable_input_require_grads()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        prep_sig = inspect.signature(prepare_model_for_kbit_training).parameters
        prep_kwargs = dict(use_gradient_checkpointing=True)
        if "activation_checkpointing_kwargs" in prep_sig:
            prep_kwargs["activation_checkpointing_kwargs"] = {"use_reentrant": False}
        else:
            prep_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
        model = prepare_model_for_kbit_training(model, **prep_kwargs)
else:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

FREEZE_PREFIXES = ["visual", "vision_tower", "merger"]
frozen_count = 0
for name, param in model.named_parameters():
    if any(tok in name for tok in FREEZE_PREFIXES):
        param.requires_grad = False
        frozen_count += 1

model.config.use_cache = False
if hasattr(model.config, "text_config") and model.config.text_config is not None:
    model.config.text_config.use_cache = False

print(f"Loaded {type(model).__name__} in {QUANT_MODE} mode. Explicitly frozen multimodal tensors: {frozen_count}")
print(f"Allocated VRAM after load: {torch.cuda.memory_allocated() / (1024**2):.1f} MiB")

In [ ]:
# @title Cell 5: Configure Text-Only LoRA Adapter
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    exclude_modules=["visual", "vision_tower", "merger"],
)

model = get_peft_model(model, lora_config)

# Ensure all trainable LoRA weights are FP32 (required by FP16 GradScaler on T4 sm_75)
cast_count = 0
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
        cast_count += 1
if cast_count:
    print(f"Cast {cast_count} trainable LoRA tensors to torch.float32 for GradScaler compatibility.")

model.print_trainable_parameters()

In [ ]:
# @title Cell 6: Train with trl.SFTTrainer (Completion-Only Loss)
import inspect
import time
from trl import SFTConfig, SFTTrainer

is_test_run = MAX_STEPS > 0
sft_sig = inspect.signature(SFTConfig.__init__).parameters

cfg_kwargs = dict(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=1 if is_test_run else 5,
    logging_steps=1 if is_test_run else 5,
    eval_strategy="steps" if is_test_run else "epoch",
    eval_steps=MAX_STEPS if is_test_run else None,
    save_strategy="no" if is_test_run else "epoch",
    load_best_model_at_end=not is_test_run,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    optim="paged_adamw_8bit",
    seed=SEED,
    report_to="none",
)

if "activation_checkpointing" in sft_sig:
    cfg_kwargs["activation_checkpointing"] = True
    cfg_kwargs["activation_checkpointing_kwargs"] = {"use_reentrant": False}
else:
    cfg_kwargs["gradient_checkpointing"] = True
    cfg_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

if "max_length" in sft_sig:
    cfg_kwargs["max_length"] = MAX_LENGTH
elif "max_seq_length" in sft_sig:
    cfg_kwargs["max_seq_length"] = MAX_LENGTH

if "completion_only_loss" in sft_sig:
    cfg_kwargs["completion_only_loss"] = True

# Remove None entries that some transformers versions reject
cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if v is not None}
sft_config = SFTConfig(**cfg_kwargs)

trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# Ensure all trainable LoRA weights are FP32 after SFTTrainer init (required by FP16 GradScaler on T4 sm_75)
cast_n = 0
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
        p.grad = None
        cast_n += 1
if cast_n:
    print(f"Cast {cast_n} trainable LoRA tensors to torch.float32 for GradScaler compatibility.")

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
train_result = trainer.train()
elapsed_s = time.perf_counter() - t0
peak_vram_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

eval_metrics = trainer.evaluate()
print(f"Training finished in {elapsed_s:.1f}s | train_loss={train_result.training_loss:.4f} | eval_loss={eval_metrics.get('eval_loss', float('nan')):.4f} | peak_vram={peak_vram_mib:.1f} MiB")

In [ ]:
# @title Cell 7: Save Best LoRA Adapter & Training Metrics
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metrics_payload = {
    **HW_META,
    "train_loss": float(train_result.training_loss),
    "eval_metrics": {k: float(v) if isinstance(v, (int, float)) else v for k, v in eval_metrics.items()},
    "elapsed_seconds": round(elapsed_s, 2),
    "peak_vram_mib": round(peak_vram_mib, 2),
    "log_history": trainer.state.log_history,
}
metrics_path = ADAPTER_DIR / "train_metrics.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")

saved_files = sorted(p.name for p in ADAPTER_DIR.iterdir() if p.is_file())
print("Saved adapter files:", saved_files)
print("Metrics written to:", metrics_path)

In [ ]:
# @title Cell 8: Post-Training Sanity Check (English, Taglish, 2028 Abstention Trap)
model.eval()
model.config.use_cache = True
if hasattr(model.config, "text_config") and model.config.text_config is not None:
    model.config.text_config.use_cache = True

def format_sanity_prompt(user_query: str) -> str:
    sys_msg = (
        "You are a precise Philippine election law and COMELEC operations assistant. "
        "Answer strictly from the 1987 Philippine Constitution and COMELEC Resolutions No. 11076 "
        "and No. 11086 (governing the 12 May 2025 National, Local, and BARMM Parliamentary Elections). "
        "Always cite exact Articles, Sections, Annexes, or Form numbers, and refuse to fabricate rules outside the documented scope."
    )
    return (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\n{user_query}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

SANITY_QUERIES = [
    ("English Rule Citation", "Under COMELEC Resolution No. 11076, SEC. 15, who may appoint watchers and what are their rights?"),
    ("Taglish Query", "Ano ang itinatakda ng 1987 Constitution Article V Section 1 tungkol sa karapatang bumoto?"),
    ("2028 Abstention Trap", "What is the official election date and voting schedule for the May 2028 Philippine Presidential Elections?"),
]

LEAK_TOKENS = ["<think>", "</think>"]
EOS_IDS = [248044, 248046]

for label, q in SANITY_QUERIES:
    prompt = format_sanity_prompt(q)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            eos_token_id=EOS_IDS,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    leaked = any(tok in reply for tok in LEAK_TOKENS)
    print("=" * 80)
    print(f"[{label}] Q: {q}")
    print(f"Thought-channel leak: {leaked}")
    print(reply)